# Classification with built-in Galaxy10 network

Pipeline is inspired by [astroNN](https://github.com/henrysky/astroNN/blob/master/demo_tutorial/galaxy10/Galaxy10_Tutorial.ipynb).

In [ ]:
# install requirements

# if connected to colab

if 'google.colab' in str(get_ipython()):
  print('running on colab')
  !git clone https://github.com/ongarileonardo/galaxy_class.git
  %cd galaxy_class

%pip install -r requirements.txt

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

# import everything we need first
from keras import utils
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

from astroNN.models import Galaxy10CNN
from astroNN.datasets import load_galaxy10sdss
from astroNN.datasets.galaxy10sdss import galaxy10cls_lookup, galaxy10_confusion

In [ ]:
# To load images and labels (will download automatically at the first time)
# First time downloading location will be ~/.astroNN/datasets/
images, labels = load_galaxy10sdss()

In [ ]:
labels

In [ ]:
counts = np.bincount(labels)
plt.bar(range(len(counts)), counts)
plt.xlabel("Class")
plt.ylabel("Occurrence")
plt.xticks(range(len(counts)))
plt.title("Class Distribution")
plt.show()

In [ ]:
# To convert the labels to categorical 10 classes
labels = utils.to_categorical(labels, 10)

# (one hot encoding)
labels

In [ ]:
# # Select 10 of the images to inspect
# img = None
# plt.ion()
# print("===================Data Inspection===================")
# for counter, i in enumerate(
#     range(np.random.randint(0, labels.shape[0], size=10).shape[0])
# ):
#     img = plt.imshow(images[i])
#     plt.title(
#         f"Class {np.argmax(labels[i])}: {galaxy10cls_lookup(labels[i])} \n Random Demo images {counter+1} of 10"
#     )
#     plt.draw()
#     plt.pause(2.0)
# plt.close("all")
# print("===============Data Inspection Finished===============")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

idx = np.random.randint(0, labels.shape[0], size=10)

for ax, i in zip(axes.flat, idx):
    ax.imshow(images[i])
    ax.set_title(f"Class {np.argmax(labels[i])}\n{galaxy10cls_lookup(labels[i])}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
# plt.savefig("galaxies.pdf")
plt.show()

In [ ]:
# def center_crop(img, crop_size):
#     h, w = img.shape[:2]
#     start_h = (h - crop_size) // 2
#     start_w = (w - crop_size) // 2
#     return img[start_h:start_h + crop_size, start_w:start_w + crop_size]

# crop_size = 69
# images_small = np.array([center_crop(img, crop_size) for img in images])
# print(images_small.shape, images_small.nbytes / 1e9, "GB")

In [ ]:
from PIL import Image
import numpy as np

images_gray = np.array([
    np.array(Image.fromarray(img).convert('L')) for img in images
])

print(images_gray.shape)  

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# idx = np.random.randint(0, labels.shape[0], size=10)

for ax, i in zip(axes.flat, idx):
    ax.imshow(images_gray[i], cmap="gray")
    ax.set_title(f"Class {np.argmax(labels[i])}\n{galaxy10cls_lookup(labels[i])}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("images shape:", images_gray.shape, images_gray.dtype)
print("labels shape:", labels.shape, labels.dtype)
print("Estimated memory (GB):", images.nbytes / 1e9)

import psutil
print("RAM available (GB):", psutil.virtual_memory().available / 1e9)
print("RAM total (GB):", psutil.virtual_memory().total / 1e9)

In [ ]:
del images

In [ ]:
# Split the dataset into training set and testing set
train_idx, test_idx = train_test_split(np.arange(labels.shape[0]), test_size=0.1)
train_images, train_labels, test_images, test_labels = (
    images_gray[train_idx],
    labels[train_idx],
    images_gray[test_idx],
    labels[test_idx],
)

del images_gray, labels

In [ ]:
train_images = train_images.astype(np.float32, copy=False)

In [ ]:
train_labels = train_labels.astype(np.float32, copy=False)

In [ ]:
test_images = test_images.astype(np.float32, copy=False)

In [ ]:
test_labels = test_labels.astype(np.float32, copy=False)

Glaxy10CNN is a simple 4 layered convolutional neural network consisted of 2 convolutional layers and 2 dense layers

In [ ]:
# To create a neural network instance
galaxy10net = Galaxy10CNN()

# set maximium epochs the neural network can run, set 5 to get quick result
galaxy10net.max_epochs = 20

# To train the neural net
# astroNN will normalize the data by default
galaxy10net.fit(train_images, train_labels)   

# print model summary 
galaxy10net.keras_model.summary()

In [ ]:
# After the training, you can test the neural net performance
# Please notice predicted_labels are labels predicted from neural network. test_labels are ground truth from the dataset
predicted_labels = galaxy10net.predict(test_images)

# Convert predicted_labels to class
prediction_class = np.argmax(predicted_labels, axis=1)

# Convert test_labels to class
test_class = np.argmax(test_labels, axis=1)

# Prepare a confusion matrix
confusion_matrix = np.zeros((10, 10))

# create the confusion matrix
for counter, i in enumerate(prediction_class):
    confusion_matrix[i, test_class[counter]] += 1

# Plot the confusion matrix
galaxy10_confusion(confusion_matrix)